In [142]:
import pandas as pd
import os

In [143]:
provincias = {
    "Madrid":"../data/raw/Madrid",
    "Barcelona":"../data/raw/Barcelona",
    "Valencia":"../data/raw/Valencia"
}
dfs_por_provincia = {
    "Madrid": [],
    "Barcelona": [],
    "Valencia": []
}

for provincia, carpeta in provincias.items():
    for archivo in os.listdir(carpeta):
        if archivo.endswith(".csv"):
            ruta = os.path.join(carpeta, archivo)
            df = pd.read_csv(ruta, encoding="latin1", sep=";", decimal=",",thousands=".")
            df["Provincia"] = provincia
            dfs_por_provincia[provincia].append(df)


In [144]:
df_madrid = pd.concat(dfs_por_provincia["Madrid"])
df_barcelona = pd.concat(dfs_por_provincia["Barcelona"])
df_valencia = pd.concat(dfs_por_provincia["Valencia"])


In [145]:
df_barcelona = df_barcelona.copy()
df_barcelona["Geografía"].head(40)

#Valores unicos ordenados
sorted(df_barcelona["Geografía"].unique())
#Numero de filas con dos o mas espacios seguidos
df_barcelona["Geografía"].str.contains(r"\s{2,}").sum()
#Numero de filas que comienzan con un espacio
df_barcelona["Geografía"].str.startswith(" ").sum()
#Frecuencias de los valores
df_barcelona["Geografía"].str.lower().value_counts()
#Normaliza espacios y luego cuenta(ver si hay variantes del mismo muncipio)
df_barcelona["Geografía"].str.strip().str.lower().value_counts()
#Estadisticas de longitud de los str
df_barcelona["Geografía"].str.len().describe().T
#Cuenta valores nulos
df_barcelona["Geografía"].isna().sum()
#Cuenta strings vacios que no son nulos
df_barcelona["Geografía"].eq("").sum()
#str que empiezan con:
df_cp = df_barcelona[~df_barcelona["Geografía"].str.startswith("- Municipio de")]
#Detectamos simbolos extraños       
df_barcelona["Geografía"].str.contains(r"[-()_/]").sum() #->472
#Detectamos los valores con estos simbolos
df_barcelona[df_barcelona["Geografía"].str.contains(r"[-()_/]")]["Geografía"].unique()



<StringArray>
[                    '- Municipio de Badalona',
           '- Municipio de Barberà del Vallès',
                    '- Municipio de Barcelona',
                '- Municipio de Castelldefels',
        '- Municipio de Cerdanyola del Vallès',
        '- Municipio de Cornellà de Llobregat',
       '- Municipio de Esplugues de Llobregat',
                         '- Municipio de Gavà',
                   '- Municipio de Granollers',
 '- Municipio de Hospitalet de Llobregat (L')',
                     '- Municipio de Igualada',
                      '- Municipio de Manresa',
                       '- Municipio de Mataró',
            '- Municipio de Mollet del Vallès',
            '- Municipio de Montcada i Reixac',
       '- Municipio de Prat de Llobregat (El)',
                     '- Municipio de Ripollet',
                         '- Municipio de Rubí',
                     '- Municipio de Sabadell',
          '- Municipio de Sant Adrià de Besòs',
        '- Municipio de Sa

In [146]:
df_valencia["Periodos:"].value_counts()

Periodos:
enero-marzo 2025         1349
enero-junio 2025         1349
enero-septiembre 2025    1349
Enero-junio 2023         1311
enero-marzo 2024         1311
Enero-marzo 2023         1311
enero-diciembre 2023     1311
enero-diciembre 2024     1311
enero-septiembre 2024    1311
enero-septiembre 2023    1311
enero-junio 2024         1311
Enero-diciembre 2022     1292
Enero-marzo 2021         1020
Enero-marzo 2022         1020
Enero-septiembre 2022    1020
Enero-junio 2021         1020
Enero-junio 2022         1020
Enero-diciembre 2021     1020
Enero-septiembre 2021    1020
Enero-diciembre 2020      465
Enero-diciembre 2019      465
Enero-marzo 2020          465
Enero-marzo 2019          465
Enero-septiembre 2020     465
Enero-septiembre 2019     465
Enero-junio 2020          465
Enero-diciembre 2018      465
Enero-septiembre 2018     434
Enero-septiembre 2017     434
Enero-marzo 2017          434
Enero-marzo 2018          434
Enero-diciembre 2017      434
Enero-Septiembre 2016     144


In [147]:
df_barcelona.duplicated().sum()
df_valencia.duplicated().sum()
df_madrid.duplicated().sum()

np.int64(0)

In [148]:
print(df_madrid["Periodos:"].unique())

<StringArray>
[     'Enero-junio 2020',      'Enero-junio 2019',      'Enero-marzo 2020',
  'Enero-diciembre 2018', 'enero-septiembre 2024',      'Enero-junio 2016',
 'enero-septiembre 2023',      'Enero-marzo 2023', 'Enero-septiembre 2020',
 'Enero-septiembre 2021',      'Enero-junio 2022',  'enero-diciembre 2023',
      'Enero-marzo 2021',      'Enero-marzo 2019',  'Enero-diciembre 2017',
      'enero-marzo 2024',  'Enero-diciembre 2020',  'Enero-diciembre 2021',
  'enero-diciembre 2024',  'Enero-diciembre 2019',      'enero-junio 2024',
      'enero-marzo 2025',      'Enero-junio 2023',      'Enero-marzo 2017',
 'enero-septiembre 2025',  'Enero-Diciembre 2016',      'Enero-marzo 2022',
      'enero-junio 2025',      'Enero-junio 2018', 'Enero-septiembre 2019',
      'Enero-marzo 2018',      'Enero-junio 2017', 'Enero-Septiembre 2016',
      'Enero-junio 2021', 'Enero-septiembre 2018', 'Enero-septiembre 2022',
  'Enero-diciembre 2022', 'Enero-septiembre 2017',      'Enero-marzo 2016'

In [149]:
def limpiar_geografia_general(df):

    df = df.copy()

    #Eliminar códigos postales (primeros 5 caracteres numéricos)
    df["Geografía"] = df["Geografía"].apply(lambda x: x[5:] if isinstance(x,str) and x[:5].isdigit() else x)
    
    #Eliminar el prefijo "- Municipio de"
    prefijos = ["- municipio de", 
                "-municipio de", 
                "- municipo de", 
                "-municipo de", 
                "municipio de", 
                "municipo de"]
    
    for p in prefijos:
        df["Geografía"] = df["Geografía"].str.replace(p,"", case=False, regex=False)
        
    #Eliminar espacios al inicio y final
    df["Geografía"] = df["Geografía"].str.strip()
    
    #Corrección puntual detectada en Madrid
    df["Geografía"] = df["Geografía"].replace("Rozas de Madrid (Las)","Las Rozas de Madrid")

    #Nos deshacemos de 2 provincias que no tienen que estar en Madrid
    df = df[~df["Geografía"].isin(["Provincia de ALMERÍA", "ANDALUCÍA"])]

    return df

df_limpieza_geografia = limpiar_geografia_general(df_valencia)
df_limpieza_geografia

,Geografía,Tipología penal,Periodos:,Total,Provincia
0,Provincia de ALICANTE/ALACANT,1.-Homicidios dolosos y asesinatos consumados,Enero-marzo 2021,0,Valencia
1,Provincia de ALICANTE/ALACANT,2.-Homicidios dolosos y asesinatos en grado te...,Enero-marzo 2021,10,Valencia
2,Provincia de ALICANTE/ALACANT,3.-Delitos graves y menos graves de lesiones y...,Enero-marzo 2021,103,Valencia
3,Provincia de ALICANTE/ALACANT,4.-Secuestro,Enero-marzo 2021,1,Valencia
4,Provincia de ALICANTE/ALACANT,5.-Delitos contra la libertad e indemnidad sexual,Enero-marzo 2021,144,Valencia
...,...,...,...,...,...
460,Valencia,8.-Hurtos,Enero-diciembre 2018,19145,Valencia
461,Valencia,9.-Sustracciones de vehículos,Enero-diciembre 2018,846,Valencia
462,Valencia,10.-Tráfico de drogas,Enero-diciembre 2018,322,Valencia
463,Valencia,Resto de infracciones penales,Enero-diciembre 2018,21412,Valencia


In [150]:
def limpiar_tipologia_penal(df):
    #Eliminamos tipología penal basura
    patron_basura = df["Tipología penal"].str.contains(r"TOTAL|Resto|I\.|II\.|III\.|EU|ciber|informátic",case=False, na=False)

    return df[~patron_basura].copy()

df_tipologia_penal_limpia = limpiar_tipologia_penal(df_limpieza_geografia)
df_tipologia_penal_limpia

,Geografía,Tipología penal,Periodos:,Total,Provincia
0,Provincia de ALICANTE/ALACANT,1.-Homicidios dolosos y asesinatos consumados,Enero-marzo 2021,0,Valencia
1,Provincia de ALICANTE/ALACANT,2.-Homicidios dolosos y asesinatos en grado te...,Enero-marzo 2021,10,Valencia
2,Provincia de ALICANTE/ALACANT,3.-Delitos graves y menos graves de lesiones y...,Enero-marzo 2021,103,Valencia
3,Provincia de ALICANTE/ALACANT,4.-Secuestro,Enero-marzo 2021,1,Valencia
4,Provincia de ALICANTE/ALACANT,5.-Delitos contra la libertad e indemnidad sexual,Enero-marzo 2021,144,Valencia
...,...,...,...,...,...
458,Valencia,"7.- Robos con fuerza en domicilios, establecim...",Enero-diciembre 2018,2767,Valencia
459,Valencia,7.1.-Robos con fuerza en domicilios,Enero-diciembre 2018,1980,Valencia
460,Valencia,8.-Hurtos,Enero-diciembre 2018,19145,Valencia
461,Valencia,9.-Sustracciones de vehículos,Enero-diciembre 2018,846,Valencia


In [151]:
mapa_normalizacion_valencia = {
    "Alboraia/Alboraya": "Alboraia / Alboraya",
    "Alboraya": "Alboraia / Alboraya",
    "Almazora/Almassora": "Almazora/Almassora",
    "Almassora": "Almazora/Almassora",
    "Sagunto/Sagunt": "Sagunto / Sagunt",
    "Valencia": "Valencia",
    "València": "Valencia"}


def normalizacion_municipios(df, mapa_municipios):
    
    df["Geografía"] = df["Geografía"].replace(mapa_municipios)

    return df

df_municipios_normalizados = normalizacion_municipios(df_tipologia_penal_limpia,mapa_normalizacion_valencia)
df_municipios_normalizados



,Geografía,Tipología penal,Periodos:,Total,Provincia
0,Provincia de ALICANTE/ALACANT,1.-Homicidios dolosos y asesinatos consumados,Enero-marzo 2021,0,Valencia
1,Provincia de ALICANTE/ALACANT,2.-Homicidios dolosos y asesinatos en grado te...,Enero-marzo 2021,10,Valencia
2,Provincia de ALICANTE/ALACANT,3.-Delitos graves y menos graves de lesiones y...,Enero-marzo 2021,103,Valencia
3,Provincia de ALICANTE/ALACANT,4.-Secuestro,Enero-marzo 2021,1,Valencia
4,Provincia de ALICANTE/ALACANT,5.-Delitos contra la libertad e indemnidad sexual,Enero-marzo 2021,144,Valencia
...,...,...,...,...,...
458,Valencia,"7.- Robos con fuerza en domicilios, establecim...",Enero-diciembre 2018,2767,Valencia
459,Valencia,7.1.-Robos con fuerza en domicilios,Enero-diciembre 2018,1980,Valencia
460,Valencia,8.-Hurtos,Enero-diciembre 2018,19145,Valencia
461,Valencia,9.-Sustracciones de vehículos,Enero-diciembre 2018,846,Valencia


In [152]:
df_municipios_normalizados[(df_municipios_normalizados["Provincia"] == "Valencia") &
                           ~(df_municipios_normalizados["Geografía"].isin([
        "Alaquàs",
        "Alboraia / Alboraya",
        "Aldaia",
        "Alfafar",
        "Algemesí",
        "Alzira",
        "Bétera",
        "Burjassot",
        "Carcaixent",
        "Catarroja",
        "Cullera",
        "Gandia",
        "Llíria",
        "Manises",
        "Mislata",
        "Moncada",
        "Oliva",
        "Ontinyent",
        "Paiporta",
        "Paterna",
        "Picassent",
        "Pobla de Vallbona (la)",
        "Puçol",
        "Quart de Poblet",
        "Requena",
        "Riba-roja de Túria",
        "Sagunto / Sagunt",
        "Silla",
        "Sueca",
        "Torrent",
        "Valencia",
        "Xàtiva",
        "Xirivella"]))]["Geografía"].unique()

<StringArray>
[                  'Provincia de ALICANTE/ALACANT',
                                     'Alcoy/Alcoi',
                                  'L'Alfàs del Pi',
                                'Alicante/Alacant',
                                        'Almoradí',
                                           'Altea',
                                            'Aspe',
                                        'Benidorm',
                                            'Calp',
                                   'Campello (el)',
                                      'Crevillent',
                                           'Dénia',
                                       'Elche/Elx',
                                            'Elda',
                                             'Ibi',
                                     'Jávea/Xàbia',
                                        'Mutxamel',
                                         'Novelda',
                                        'Orihuela'

In [153]:
def filtrar_municipios(df):
    
    valencia_municipios = [
        "Alaquàs",
        "Alboraia / Alboraya",
        "Aldaia",
        "Alfafar",
        "Algemesí",
        "Alzira",
        "Bétera",
        "Burjassot",
        "Carcaixent",
        "Catarroja",
        "Cullera",
        "Gandia",
        "Llíria",
        "Manises",
        "Mislata",
        "Moncada",
        "Oliva",
        "Ontinyent",
        "Paiporta",
        "Paterna",
        "Picassent",
        "Pobla de Vallbona (la)",
        "Puçol",
        "Quart de Poblet",
        "Requena",
        "Riba-roja de Túria",
        "Sagunto / Sagunt",
        "Silla",
        "Sueca",
        "Torrent",
        "Valencia",
        "Xàtiva",
        "Xirivella"]

    alicante_municipios = [
        "Alcoy/Alcoi",
        "Alicante/Alacant",
        "Benidorm",
        "Elche/Elx",
        "Elda",
        "Orihuela",
        "San Vicente del Raspeig / Sant Vicent del Raspeig",
        "Torrevieja",
        "Dénia",
        "Petrer",
        "Santa Pola",
        "Villajoyosa / Vila Joiosa (la)",
        "Villena",
        "L'Alfàs del Pi",
        "Almoradí",
        "Altea",
        "Aspe",
        "Calp",
        "Campello (el)",
        "Crevillent",
        "Ibi",
        "Jávea / Xàbia",
        "Mutxamel",
        "Novelda",
        "Pilar de la Horadada",
        "Sant Joan d'Alacant"]
        
    castellon_municipios = [
        "Castellón de la Plana / Castelló de la Plana",
        "Vila-Real",
        "Borriana / Burriana",
        "Vall d'Uixó (la)",
        "Almazora / Almassora",
        "Benicarló",
        "Onda",
        "Vinaròs",
        "Benicasim / Benicàssim"]
    
    es_comunidad_valenciana = df["Provincia"].iloc[0] == "Valencia"

    if es_comunidad_valenciana:
        
        df = df[df["Geografía"].isin(valencia_municipios + alicante_municipios + castellon_municipios)] 

        df.loc[df["Geografía"].isin(alicante_municipios), "Provincia"] = "Alicante"
        df.loc[df["Geografía"].isin(castellon_municipios), "Provincia"] = "Castellón"
    else:
        pass

    return df

df_municipios_filtrados = filtrar_municipios(df_municipios_normalizados)
df_municipios_filtrados["Provincia"].value_counts()

Provincia
Valencia     8982
Alicante     7002
Castellón    1074
Name: count, dtype: int64

In [154]:
df_municipios_normalizados["Tipología penal"].value_counts()


Tipología penal
5.1.-Agresión sexual con penetración                                          1712
7.1.-Robos con fuerza en domicilios                                           1712
1.-Homicidios dolosos y asesinatos consumados                                  879
2.-Homicidios dolosos y asesinatos en grado tentativa                          879
3.-Delitos graves y menos graves de lesiones y riña tumultuaria                879
4.-Secuestro                                                                   879
5.-Delitos contra la libertad e indemnidad sexual                              879
6.-Robos con violencia e intimidación                                          879
7.- Robos con fuerza en domicilios, establecimientos y otras instalaciones     879
8.-Hurtos                                                                      879
9.-Sustracciones de vehículos                                                  879
10.-Tráfico de drogas                                                  

In [155]:
def normalizar_texto_tipologia_penal(df):
    df = df.copy()
    df["Tipología penal"] = (df["Tipología penal"]
                             .str.strip()
                             .str.lower()
                             .str.replace(r"\s+", " ", regex=True) #normaliza espacios
                             .str.strip()
                             )
    df["Tipología penal"] = df["Tipología penal"].str.title()
    return df

df_tipologia_penal_normalizado = normalizar_texto_tipologia_penal(df_municipios_filtrados)
print(df_tipologia_penal_normalizado["Periodos:"].value_counts())

Periodos:
enero-marzo 2025         720
enero-junio 2025         720
enero-septiembre 2025    720
Enero-junio 2023         708
enero-marzo 2024         708
Enero-marzo 2023         708
enero-diciembre 2023     708
enero-diciembre 2024     708
enero-septiembre 2024    708
enero-septiembre 2023    708
enero-junio 2024         708
Enero-marzo 2021         696
Enero-marzo 2022         696
Enero-septiembre 2022    696
Enero-diciembre 2022     696
Enero-junio 2021         696
Enero-junio 2022         696
Enero-diciembre 2021     696
Enero-septiembre 2021    696
Enero-septiembre 2018    276
Enero-diciembre 2020     276
Enero-diciembre 2019     276
Enero-septiembre 2017    276
Enero-marzo 2020         276
Enero-marzo 2019         276
Enero-septiembre 2020    276
Enero-septiembre 2019    276
Enero-marzo 2017         276
Enero-marzo 2018         276
Enero-junio 2020         276
Enero-diciembre 2017     276
Enero-diciembre 2018     276
Enero-Septiembre 2016     26
Enero-Diciembre 2016      26
Ener

In [156]:
df_tipologia_penal_normalizado["Tipología penal"].unique()


<StringArray>
[                             '1.-Homicidios Dolosos Y Asesinatos Consumados',
                      '2.-Homicidios Dolosos Y Asesinatos En Grado Tentativa',
            '3.-Delitos Graves Y Menos Graves De Lesiones Y Riña Tumultuaria',
                                                               '4.-Secuestro',
                          '5.-Delitos Contra La Libertad E Indemnidad Sexual',
                                       '5.1.-Agresión Sexual Con Penetración',
                                      '6.-Robos Con Violencia E Intimidación',
 '7.- Robos Con Fuerza En Domicilios, Establecimientos Y Otras Instalaciones',
                                        '7.1.-Robos Con Fuerza En Domicilios',
                                                                  '8.-Hurtos',
                                              '9.-Sustracciones De Vehículos',
                                                      '10.-Tráfico De Drogas',
                                      

In [157]:
def generalizacion_tipologia_penal(df):
    '''mapa_delitos = {
    # Homicidios
    "1.-Homicidios Dolosos Y Asesinatos Consumados": "1 Homicidios Dolosos Y Asesinatos Consumados",
    "1. Homicidios Dolosos Y Asesinatos Consumados": "1 Homicidios Dolosos Y Asesinatos Consumados",
    "2.-Homicidios Dolosos Y Asesinatos En Grado Tentativa": "2 Homicidios Dolosos Y Asesinatos En Grado Tentativa",
    "2. Homicidios Dolosos Y Asesinatos En Grado Tentativa": "2 Homicidios Dolosos Y Asesinatos En Grado Tentativa",

    # Lesiones
    "3.-Delitos Graves Y Menos Graves De Lesiones Y Riña Tumultuaria": "3 Lesiones",
    "3. Delitos Graves Y Menos Graves De Lesiones Y Riña Tumultuaria": "3 Lesiones",

    # Secuestro
    "4.-Secuestro": "4 Secuestro",
    "4. Secuestro": "4 Secuestro",

    # Delitos sexuales
    "5.-Delitos Contra La Libertad E Indemnidad Sexual": "5a Delitos Contra La Libertad E Indemnidad Sexual",
    "5. Delitos Contra La Libertad Sexual": "5b Delitos Contra La Libertad Sexual",
    "5.1.-Agresión Sexual Con Penetración": "5.1 Agresión Sexual Con Penetración",
    "5.2.-Resto De Delitos Contra La Libertad E Indemnidad Sexual": "5.2a Resto De Delitos Contra La Libertad E Indemnidad Sexual",
    "5.2.-Resto De Delitos Contra La Libertad Sexual": "5.2b Resto De Delitos Contra La Libertad Sexual",

    # Robos con violencia
    "6.-Robos Con Violencia E Intimidación": "6 Robos Con Violencia",
    "6. Robos Con Violencia E Intimidación": "6 Robos Con Violencia",

    # Robos con fuerza
    "7.- Robos Con Fuerza En Domicilios, Establecimientos Y Otras Instalaciones": "7 Robos Con Fuerza",
    "7. Robos Con Fuerza En Domicilios, Establecimientos Y Otras Instalaciones": "7 Robos Con Fuerza",
    "7.1.-Robos Con Fuerza En Domicilios": "7 Robos Con Fuerza",

    # Hurtos
    "8.-Hurtos": "8 Hurtos",
    "8. Hurtos": "8 Hurtos",

    # Sustracciones
    "9.-Sustracciones De Vehículos": "9 Sustraccion De Vehiculos",
    "9. Sustracciones De Vehículos": "9 Sustraccion De Vehiculos",

    # Tráfico de drogas
    "10.-Tráfico De Drogas": "10 Trafico De Drogas",
    "10. Tráfico De Drogas": "10 Trafico De Drogas",

    # Daños (categoría antigua)
    "7.-Daños": "11 Daños"
    }'''

    mapa_delitos = {
    # Homicidios
    "1.-Homicidios Dolosos Y Asesinatos Consumados": "Homicidios",
    "1. Homicidios Dolosos Y Asesinatos Consumados": "Homicidios",
    "2.-Homicidios Dolosos Y Asesinatos En Grado Tentativa": "Homicidios",
    "2. Homicidios Dolosos Y Asesinatos En Grado Tentativa": "Homicidios",

    # Lesiones
    "3.-Delitos Graves Y Menos Graves De Lesiones Y Riña Tumultuaria": "Lesiones",
    "3. Delitos Graves Y Menos Graves De Lesiones Y Riña Tumultuaria": "Lesiones",

    # Secuestro
    "4.-Secuestro": "Secuestro",
    "4. Secuestro": "Secuestro",

    # Delitos sexuales
    "5.-Delitos Contra La Libertad E Indemnidad Sexual": "Delitos Sexuales",
    "5. Delitos Contra La Libertad Sexual": "Delitos Sexuales",
    "5.1.-Agresión Sexual Con Penetración": "Delitos Sexuales",
    "5.2.-Resto De Delitos Contra La Libertad E Indemnidad Sexual": "Delitos Sexuales",
    "5.2.-Resto De Delitos Contra La Libertad Sexual": "Delitos Sexuales",

    # Robos con violencia
    "6.-Robos Con Violencia E Intimidación": "Robos Con Violencia",
    "6. Robos Con Violencia E Intimidación": "Robos Con Violencia",

    # Robos con fuerza
    "7.- Robos Con Fuerza En Domicilios, Establecimientos Y Otras Instalaciones": "Robos Con Fuerza",
    "7. Robos Con Fuerza En Domicilios, Establecimientos Y Otras Instalaciones": "Robos Con Fuerza",
    "7.1.-Robos Con Fuerza En Domicilios": "Robos Con Fuerza",

    # Hurtos
    "8.-Hurtos": "Hurtos",
    "8. Hurtos": "Hurtos",

    # Sustracciones
    "9.-Sustracciones De Vehículos": "Sustraccion De Vehiculos",
    "9. Sustracciones De Vehículos": "Sustraccion De Vehiculos",

    # Tráfico de drogas
    "10.-Tráfico De Drogas": "Trafico De Drogas",
    "10. Tráfico De Drogas": "Trafico De Drogas",

    # Daños (categoría antigua)
    "7.-Daños": "Daños"
    }


    df["Delitos"] = df["Tipología penal"].map(mapa_delitos)
    
    return df

df_generalizacion_delitos = generalizacion_tipologia_penal(df_tipologia_penal_normalizado)
df_generalizacion_delitos
print(df_generalizacion_delitos["Periodos:"].unique())

<StringArray>
[     'Enero-marzo 2021', 'Enero-Septiembre 2016',      'enero-marzo 2025',
 'Enero-septiembre 2018',      'Enero-junio 2023',      'enero-junio 2025',
      'Enero-marzo 2022',  'Enero-diciembre 2020',  'Enero-diciembre 2019',
 'Enero-septiembre 2022', 'Enero-septiembre 2017',      'Enero-marzo 2020',
      'Enero-marzo 2019',      'enero-marzo 2024', 'Enero-septiembre 2020',
 'enero-septiembre 2025',      'Enero-marzo 2023',  'Enero-diciembre 2022',
  'Enero-Diciembre 2016', 'Enero-septiembre 2019',      'Enero-marzo 2017',
  'enero-diciembre 2023',      'Enero-junio 2021',      'Enero-marzo 2018',
  'enero-diciembre 2024', 'enero-septiembre 2024',      'Enero-junio 2020',
 'enero-septiembre 2023',      'Enero-junio 2022',      'enero-junio 2024',
  'Enero-diciembre 2021',      'Enero-marzo 2016',  'Enero-diciembre 2017',
 'Enero-septiembre 2021',  'Enero-diciembre 2018']
Length: 35, dtype: str


In [158]:
sorted(df_generalizacion_delitos["Periodos:"].unique())

df_generalizacion_delitos["Periodos:"].str.startswith(" ").sum()

df_generalizacion_delitos["Periodos:"].str.strip().str.lower().value_counts()

df_generalizacion_delitos["Periodos:"].str.len().describe().T

df_generalizacion_delitos["Periodos:"].isna().sum()

df_generalizacion_delitos["Periodos:"].eq("").sum()


#Que valores de la columna "Periodos:" no cumplen el formato correcto:
#Que empiecen con una palabra
#Guion obligatorio
#Otra palabra
#Espacio
#Un año de 4 digitos
df_generalizacion_delitos[
    ~df_generalizacion_delitos["Periodos:"].str.match(r"^[A-Za-zÁÉÍÓÚáéíóúñÑ]+-[A-Za-zÁÉÍÓÚáéíóúñÑ]+ \d{4}$")
]["Periodos:"].unique()
print(df.columns.tolist())


['Geografía', 'Tipología penal', 'Periodos:', 'Total', 'Provincia']


In [159]:
def normalizacion_año_trimestre(df):
    df = df.copy()
    #Normalizar texto del periodo
    periodos = (
    df["Periodos:"]
    .str.lower()
    .str.replace(r"\s*[-–]\s*", "-", regex=True)  # " - " o "–" → "-"
    .str.replace(r"\s+", " ", regex=True)
    .str.strip())
    
    #Extraccion del año
    df["Año"] = periodos.str.extract(r"(\d{4})$")
    
    #Extraer solo la parte del periodo (antes del año)
    df["Periodo"] = periodos.str.extract(r"^([a-záéíóúñ\- ]+)")
    df["Periodo"] = df["Periodo"].str.strip().str.title()

    # Filtrar solo el acumulado anual (Enero-Diciembre) 
    #df = df[df["Periodo"] == "Enero-Diciembre"]

    #Creamos Trimestres
    mapa_periodos = {
    "Enero-Marzo": 1,
    "Enero-Junio": 2,
    "Enero-Septiembre": 3,
    "Enero-Diciembre": 4}

    # Creamos una columna  para ordenar
    df['Trimestre'] = df['Periodo'].map(mapa_periodos)
    
    return df

df_periodos = normalizacion_año_trimestre(df_generalizacion_delitos)

df_periodos["Trimestre"].value_counts()


Trimestre
1    4658
3    4658
4    3938
2    3804
Name: count, dtype: int64

### Informacion
-Agrupamos y dentro de cada grupo sumamos la columna "Total". Por que?:
-Antes teniamos un mismo delito con diferentes nombres, como los categorizamos en delitos mas "generales" tenemos que sumarlos para obtener el valor real de esa categoria delictiva general

In [160]:
def agregar_duplicados_tipologias(df):
    # Incluimos 'Triemstre' para no perder la secuencia 1, 2, 3, 4
    columnas_clave = ["Geografía", "Delitos", "Año", "Periodo", "Trimestre", "Provincia"]
    
    # Sumamos los Totales (acumulados) de las categorías que ahora coinciden
    df_agrupado = df.groupby(columnas_clave, as_index=False)["Total"].sum()
    
    return df_agrupado

df_suma_tipologias = agregar_duplicados_tipologias(df_periodos)
df_suma_tipologias

,Geografía,Delitos,Año,Periodo,Trimestre,Provincia,Total
0,Alaquàs,Delitos Sexuales,2021,Enero-Diciembre,4,Valencia,4
1,Alaquàs,Delitos Sexuales,2021,Enero-Junio,2,Valencia,2
2,Alaquàs,Delitos Sexuales,2021,Enero-Marzo,1,Valencia,2
3,Alaquàs,Delitos Sexuales,2021,Enero-Septiembre,3,Valencia,2
4,Alaquàs,Delitos Sexuales,2022,Enero-Diciembre,4,Valencia,5
...,...,...,...,...,...,...,...
12808,Xàtiva,Trafico De Drogas,2024,Enero-Marzo,1,Valencia,2
12809,Xàtiva,Trafico De Drogas,2024,Enero-Septiembre,3,Valencia,2
12810,Xàtiva,Trafico De Drogas,2025,Enero-Junio,2,Valencia,3
12811,Xàtiva,Trafico De Drogas,2025,Enero-Marzo,1,Valencia,2


In [161]:
def calcular_trimestres_reales(df):

    df = df.sort_values(["Geografía","Delitos", "Año", "Trimestre"])
    df = df.reset_index(drop=True)

    #Obtenemos el acumulado de cada agrupacion
    total_agrupacion = df.groupby(["Geografía","Delitos","Año"])["Total"]

    #Desplaza los valores una fila hacia abajo dentro de cada grupo:
    #En cada fila, 'acumulado_anterior' contiene el acumulado del trimestre anterior
    #Para la fila del T2 guarda el valor que habia en el T1...
    acumulado_anterior = total_agrupacion.shift() #Empuja los valores una fila hacia abajo para que cada T vea la cigra del anterior.

    # Marcamos como inválidas las filas donde el acumulado baja respecto al trimestre anterior
    #Si el total actual es menor que el del anterior marca TRUE, es decir falso acumulativo
    df["invalido"] = df["Total"] < acumulado_anterior

    #Detectamos acumulados FALSOS:
    #Si el acumulado actual es menor que el anterior, ese valor no puede ser real
    mask = df["Total"] < acumulado_anterior 

    #Corregimos los acumulados inválidos: 
    #En las filas donde el acumulado baja, sustituimos por el acumulado anterior
    df.loc[mask, "Total"] = acumulado_anterior 

    #Calculo del valor real de cada trimestre
    #Dentro de cada grupo restamos el acumulado anterior (diff)
    #Rellenamos el NaN del T1 con su propio acumulado(fillna) 
    #Ejem: T3-T2 = valor real del T3   
    df["Valor trimestral"] = df.groupby(["Geografía", "Delitos", "Año"])["Total"].diff().fillna(df["Total"])
    
    #Eliminamos columna INVALIDO que ya no necesitamos
    df = df.drop(columns="invalido")
    return df

df_trimestres_reales = calcular_trimestres_reales(df_suma_tipologias)
df_trimestres_reales


,Geografía,Delitos,Año,Periodo,Trimestre,Provincia,Total,Valor trimestral
0,Alaquàs,Delitos Sexuales,2021,Enero-Marzo,1,Valencia,2,2.0
1,Alaquàs,Delitos Sexuales,2021,Enero-Junio,2,Valencia,2,0.0
2,Alaquàs,Delitos Sexuales,2021,Enero-Septiembre,3,Valencia,2,0.0
3,Alaquàs,Delitos Sexuales,2021,Enero-Diciembre,4,Valencia,4,2.0
4,Alaquàs,Delitos Sexuales,2022,Enero-Marzo,1,Valencia,0,0.0
...,...,...,...,...,...,...,...,...
12808,Xàtiva,Trafico De Drogas,2024,Enero-Septiembre,3,Valencia,2,0.0
12809,Xàtiva,Trafico De Drogas,2024,Enero-Diciembre,4,Valencia,4,2.0
12810,Xàtiva,Trafico De Drogas,2025,Enero-Marzo,1,Valencia,2,2.0
12811,Xàtiva,Trafico De Drogas,2025,Enero-Junio,2,Valencia,3,1.0


In [162]:
# Buscamos las filas de 2020 donde el valor trimestral salió negativo
errores_2020 = df_trimestres_reales[
    (df_trimestres_reales["Año"] == "2020") & 
    (df_trimestres_reales["Valor trimestral"] < 0)
]

# Mostramos las columnas clave para entender por qué la corrección falló
print(errores_2020[["Geografía", "Trimestre", "Total", "Valor trimestral"]])

Empty DataFrame
Columns: [Geografía, Trimestre, Total, Valor trimestral]
Index: []


In [163]:
def pipeline_provincia(df, mapa_normalizacion):
    df = limpiar_geografia_general(df)
    df = limpiar_tipologia_penal(df)

    # Normalización de nombres de municipios (si hay mapa)
    if mapa_normalizacion:
        df = normalizacion_municipios(df, mapa_normalizacion)

    df = filtrar_municipios(df)
    df = normalizar_texto_tipologia_penal(df)
    df = generalizacion_tipologia_penal(df)
    df = normalizacion_año_trimestre(df)
    
    return df


df_valencia_limpio = pipeline_provincia(df_valencia,mapa_normalizacion_valencia)

df_barcelona_limpio = pipeline_provincia(df_barcelona,{})

df_madrid_limpio = pipeline_provincia(df_madrid, {})


In [164]:
df_final = pd.concat([df_valencia_limpio, df_barcelona_limpio, df_madrid_limpio], ignore_index=True)

# 1. Agregamos duplicados (ahora que están todos los datos juntos)
df_final = agregar_duplicados_tipologias(df_final)

# 2. Calculamos los trimestres reales sobre el total limpio
df_final = calcular_trimestres_reales(df_final)

# 3. cambiamos nombres de columnas
df_final = df_final.rename(columns={"Geografía":"Municipio"})

# 4. Guardamos el resultado final
df_final.to_csv("../data/processed/df_final.csv", index=False)

df_final


,Municipio,Delitos,Año,Periodo,Trimestre,Provincia,Total,Valor trimestral
0,Alaquàs,Delitos Sexuales,2021,Enero-Marzo,1,Valencia,2,2.0
1,Alaquàs,Delitos Sexuales,2021,Enero-Junio,2,Valencia,2,0.0
2,Alaquàs,Delitos Sexuales,2021,Enero-Septiembre,3,Valencia,2,0.0
3,Alaquàs,Delitos Sexuales,2021,Enero-Diciembre,4,Valencia,4,2.0
4,Alaquàs,Delitos Sexuales,2022,Enero-Marzo,1,Valencia,0,0.0
...,...,...,...,...,...,...,...,...
35593,Xàtiva,Trafico De Drogas,2024,Enero-Septiembre,3,Valencia,2,0.0
35594,Xàtiva,Trafico De Drogas,2024,Enero-Diciembre,4,Valencia,4,2.0
35595,Xàtiva,Trafico De Drogas,2025,Enero-Marzo,1,Valencia,2,2.0
35596,Xàtiva,Trafico De Drogas,2025,Enero-Junio,2,Valencia,3,1.0


In [165]:
total_filas = len(df_final)
total_ceros = len(df_final[df_final['Valor trimestral'] == 0])
porcentaje_ceros = (total_ceros / total_filas) * 100

print(f"Tienes un {porcentaje_ceros:.2f}% de ceros en tu dataset.")

Tienes un 21.96% de ceros en tu dataset.


In [166]:
negativos = df_final[df_final['Valor trimestral'] < 0]
df_final.loc[df_final["Valor trimestral"] < 0, "Valor trimestral"] = 0